In [ ]:
#enable autoreload

%load_ext autoreload
%autoreload all

#common jupyter functions
from IPython.display import display, clear_output
from tqdm.notebook import tqdm

#configure matplotlib
%matplotlib widget
import matplotlib.pyplot as plt
plt.ioff()

import sys, os.path
importPath = os.path.abspath('../../common')
if not importPath in sys.path:
    sys.path.append(importPath)

In [ ]:
#load model and tokenizer

import torch

#define model to use
litModelPath = "../logs/potion_mtclmtlbl_words_binary-mixed_v2/lightning_logs/version_1/checkpoints/epoch=9-step=9.ckpt"

#load the ligtning wrapper, extract the model
from nns.potion_multiclass_multilabel_v2 import MultiClassMultiLabelTokenLabelerLit, Tokenizer
litModel = MultiClassMultiLabelTokenLabelerLit.load_from_checkpoint(litModelPath)
model = litModel.model
del litModel	

model.eval() #make sure we are not in training
model.to(torch.device("cuda"))

#create tokenizer
tokenizer = Tokenizer.load(model.getMaxInputLen())

In [ ]:
# #run interpretter over sample input

# #create embedder
# from nns.embedders import buildPotionEmbedderV2
# embedder = buildPotionEmbedderV2(torch.device("cuda"))

# #define text to work on
# text = "Hoholu propaganda nieko nesiskiria nuo kremlyno"
# # text = "kada tik parasus surinksit tada atjungs ir dujas. nors lygtais per dujas neina melas? ar ne?"

# #create interpreter
# from nns.intprig import Interpreter, InputProcessor

# inpproc = InputProcessor()
# intpr = Interpreter()

# inflAll = intpr.run(text=text, inpproc=inpproc, model=model, tokenizer=tokenizer, embedder=embedder)
# inflAll[0]

In [ ]:
#test how results change with changing number of iterations

#create embedder
from nns.embedders import buildPotionEmbedderV2
embedder = buildPotionEmbedderV2(torch.device("cuda"))

#define text to work on
text = "Hoholu propaganda nieko nesiskiria nuo kremlyno"
# text = "kada tik parasus surinksit tada atjungs ir dujas. nors lygtais per dujas neina melas? ar ne?"

#create interpreter
from nns.intprig import Interpreter, InputProcessor
inpproc = InputProcessor()
intpr = Interpreter()

#define step counts
numStepsAll = [100, 500, 1000, 1500, 2000, 2500, 3000, 3500, 4000, 4500, 5000, 5500, 6000, 6500, 7000, 7500, 8000]

#gather results
inflAll = []

for numSteps in numStepsAll:
    intpr.numIgSteps = numSteps
    res = intpr.run(text=text, inpproc=inpproc, model=model, tokenizer=tokenizer, embedder=embedder, normalize=False)
    inflAll.append(res[0])

#compute differences (% of larger value) between results, gather statistics
maxChange = []
minChange = []
medChange = []
avgChange = []

for stepIdx in range(len(inflAll) - 1):
    curRes = inflAll[stepIdx]
    nexRes = inflAll[stepIdx+1]

    resDiff = torch.abs((torch.abs(nexRes) - torch.abs(curRes)) / torch.maximum(torch.abs(curRes), torch.abs(nexRes)) * 100)

    max = torch.max(resDiff).item()
    min = torch.min(resDiff).item()
    med = torch.median(resDiff).item()
    avg = torch.mean(resDiff).item()

    maxChange.append(max)
    minChange.append(min)
    medChange.append(med)
    avgChange.append(avg)

#plot
x = torch.arange(len(numStepsAll)-1)
xLabels = [f"({numStepsAll[i+1]} vs {numStepsAll[i]})" for i in x]

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(x, maxChange, label="max", marker=".")
ax.plot(x, minChange, label="min", marker=".")
ax.plot(x, medChange, label="med", marker=".")
ax.plot(x, avgChange, label="avg", marker=".")

ax.set_xticks(x)
ax.set_xticklabels(xLabels)
ax.tick_params("x", rotation=45)

ax.legend()
ax.set_ylabel("abs(delta) in % of abs(larger value)")
ax.set_xlabel("steps")
ax.set_title("Change of result values")

fig.tight_layout()
fig.show()


In [ ]:
# #plot result for chosen head

# infl = inflAll[0]

# import matplotlib as mpl
# from matplotlib.axes import Axes
# from matplotlib.figure import Figure
# import matplotlib.pyplot as plt

# fig : Figure
# ax : Axes
# fig, ax = plt.subplots(figsize=(10, 10))

# mat = ax.matshow(
#     infl,
#     aspect="equal",
#     cmap = plt.get_cmap("RdYlGn")
# )
# # ax.set_yticks(
# #     ticks=range(attrs.shape[0]),
# #     labels=[f"{outputTknNames[id]}\n({txt})" for (txt, id) in zip(res.wrdTexts, res.wrdLbls)]
# # )
# ax.set_ylabel("Outputs")

# # ax.set_xticks(
# #     ticks=range(attrs.shape[1]),
# #     labels=res.wrdTexts,
# #     rotation=45
# # )
# ax.set_xlabel("Inputs")

# for r in range(infl.shape[0]):
#     for c in range(infl.shape[1]):
#         val = infl[r][c]
#         ax.text(c, r, f"{val:0.3f}", ha='center', va='center', size=8, rotation=45)

# cbar = plt.colorbar(mat)

# fig.show()